# 03 Data Split — Women's

This notebook constructs the matchup-level training dataset from historical women's tournament games and defines the cross-validation strategy.

**Key design decisions:**
- **Training target**: Tournament game outcomes (binary: did the lower-ID team win?)
- **CV strategy**: Leave-One-Season-Out (LOGO) — train on all seasons except one, validate on the held-out season's tournament
- **Women's TeamIDs**: 3000–3999 (used to filter sample submissions)
- **Exclude 2020**: No tournament was played
- **Tournament history**: Women's tournament data begins in 1998 (not 1985 like men's)
- **2026 prediction grid**: All possible women's team matchups for Stage 2 submission

**Inputs** (from S3 `01_data_joining/womens/`):
- `tourney_games.parquet`
- `tourney_seeds.parquet`
- `SampleSubmissionStage1.csv` and `SampleSubmissionStage2.csv` (from `00_data_collection/`)

**Outputs** (to S3 `03_data_split/womens/`):
1. `tourney_matchups.parquet` — all historical tournament matchups with labels and fold assignments
2. `prediction_grid_stage1.parquet` — all women's matchups from Stage 1 sample submission (2022–2025)
3. `prediction_grid_stage2.parquet` — all women's matchups from Stage 2 sample submission (2026)

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 if available, otherwise fall back to local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"{LOCAL_INPUT}{filename}")

def read_csv(filename):
    """Read CSV from S3 raw data or local."""
    try:
        return pd.read_csv(f"s3://{BUCKET}/00_data_collection/{filename}")
    except Exception:
        return pd.read_csv(f"../00_data_collection/{filename}")

def parse_submission_grid(sample_sub):
    """Parse sample submission into Season, TeamA, TeamB columns."""
    grid = sample_sub[['ID']].copy()
    parts = grid['ID'].str.split('_', expand=True)
    grid['Season'] = parts[0].astype(int)
    grid['TeamA'] = parts[1].astype(int)
    grid['TeamB'] = parts[2].astype(int)
    return grid

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "03_data_split"
INPUT_PREFIX = f"s3://{BUCKET}/01_data_joining/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_INPUT = "../01_data_joining/output/"
LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
tourney_games = read_parquet("tourney_games.parquet")
seeds = read_parquet("tourney_seeds.parquet")
sample_sub_stage1 = read_csv("SampleSubmissionStage1.csv")
sample_sub_stage2 = read_csv("SampleSubmissionStage2.csv")

print(f"Tournament games: {tourney_games.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Stage 1 submission rows: {len(sample_sub_stage1):,}")
print(f"Stage 2 submission rows: {len(sample_sub_stage2):,}")

Tournament games: (1717, 34)
Seeds: (1812, 6)
Stage 1 submission rows: 519,144
Stage 2 submission rows: 132,133


## 2. Build Tournament Matchup Dataset

Convert tournament games from winner/loser format into the submission format: `(Season, TeamID_lower, TeamID_higher)` with a binary label indicating whether the lower-ID team won.

This matches the submission format where `ID = YYYY_XXXX_YYYY` (XXXX < YYYY) and `Pred` = P(lower-ID team wins).

Women's tournament data starts in 1998.

In [6]:
# Build matchups in submission format
matchups = tourney_games[['Season', 'WTeamID', 'LTeamID']].copy()

# Assign TeamA = lower ID, TeamB = higher ID (matches submission format)
matchups['TeamA'] = matchups[['WTeamID', 'LTeamID']].min(axis=1)
matchups['TeamB'] = matchups[['WTeamID', 'LTeamID']].max(axis=1)

# Label: 1 if lower-ID team (TeamA) won, 0 otherwise
matchups['Label'] = (matchups['WTeamID'] == matchups['TeamA']).astype(int)

# Create ID in submission format
matchups['ID'] = (matchups['Season'].astype(str) + '_' + 
                  matchups['TeamA'].astype(str) + '_' + 
                  matchups['TeamB'].astype(str))

matchups = matchups[['ID', 'Season', 'TeamA', 'TeamB', 'Label']]

print(f"Tournament matchups: {matchups.shape}")
print(f"Seasons: {matchups.Season.min()} - {matchups.Season.max()}")
print(f"Label distribution: {matchups.Label.mean():.3f} (should be ~0.5)")
matchups.head()

Tournament matchups: (1717, 5)
Seasons: 1998 - 2025
Label distribution: 0.517 (should be ~0.5)


,ID,Season,TeamA,TeamB,Label
0,1998_3104_3422,1998,3104,3422,1
1,1998_3112_3365,1998,3112,3365,1
2,1998_3163_3193,1998,3163,3193,1
3,1998_3198_3266,1998,3198,3266,1
4,1998_3203_3208,1998,3203,3208,1


## 3. Assign LOGO Cross-Validation Folds

Each season is one fold. Women's tournament data starts in 1998, giving us ~27 folds (1998–2025, excluding 2020).

We also mark the Stage 1 validation seasons (2022–2025) which simulate the Kaggle leaderboard.

In [7]:
# Exclude 2020 (no tournament played)
matchups = matchups[matchups.Season != 2020].copy()

# Fold = Season (for LOGO CV)
matchups['Fold'] = matchups['Season']

# Mark Stage 1 validation years
stage1_seasons = [2022, 2023, 2024, 2025]
matchups['IsStage1Val'] = matchups['Season'].isin(stage1_seasons).astype(int)

print(f"Matchups after excluding 2020: {matchups.shape}")
print(f"Unique folds (seasons): {matchups.Fold.nunique()}")
print(f"\nGames per fold:")
fold_counts = matchups.groupby('Fold').size()
print(f"  Min: {fold_counts.min()}, Max: {fold_counts.max()}, Mean: {fold_counts.mean():.1f}")
print(f"\nStage 1 validation games: {matchups.IsStage1Val.sum()}")
print(f"Training games (non-Stage 1): {(matchups.IsStage1Val == 0).sum()}")

Matchups after excluding 2020: (1717, 7)
Unique folds (seasons): 27

Games per fold:
  Min: 63, Max: 67, Mean: 63.6

Stage 1 validation games: 268
Training games (non-Stage 1): 1449


#### Fold Summary

In [8]:
# Summary of folds
fold_summary = matchups.groupby('Fold').agg(
    Games=('Label', 'count'),
    LabelMean=('Label', 'mean'),
    IsStage1Val=('IsStage1Val', 'first')
).reset_index()

print("Fold summary:")
print(fold_summary.to_string(index=False))

Fold summary:
 Fold  Games  LabelMean  IsStage1Val
 1998     63   0.587302            0
 1999     63   0.634921            0
 2000     63   0.555556            0
 2001     63   0.476190            0
 2002     63   0.507937            0
 2003     63   0.460317            0
 2004     63   0.555556            0
 2005     63   0.555556            0
 2006     63   0.555556            0
 2007     63   0.476190            0
 2008     63   0.444444            0
 2009     63   0.587302            0
 2010     63   0.634921            0
 2011     63   0.476190            0
 2012     63   0.507937            0
 2013     63   0.634921            0
 2014     63   0.476190            0
 2015     63   0.587302            0
 2016     63   0.412698            0
 2017     63   0.444444            0
 2018     63   0.539683            0
 2019     63   0.507937            0
 2021     63   0.587302            0
 2022     67   0.417910            1
 2023     67   0.432836            1
 2024     67   0.477612 

## 4. Build Prediction Grids

Parse the sample submission files to get the prediction grids. Filter to women's matchups only (TeamIDs 3000–3999).

In [9]:
# Stage 1: 2022-2025 all possible matchups
grid_stage1 = parse_submission_grid(sample_sub_stage1)
# Filter to women's only (TeamIDs 3000-3999)
grid_stage1_womens = grid_stage1[grid_stage1.TeamA >= 3000].copy()

# Stage 2: 2026 all possible matchups
grid_stage2 = parse_submission_grid(sample_sub_stage2)
grid_stage2_womens = grid_stage2[grid_stage2.TeamA >= 3000].copy()

print(f"Stage 1 prediction grid (women's): {grid_stage1_womens.shape}")
print(f"  Seasons: {grid_stage1_womens.Season.unique()}")
print(f"  Unique teams per season: {grid_stage1_womens.groupby('Season')['TeamA'].nunique().to_dict()}")
print(f"\nStage 2 prediction grid (women's): {grid_stage2_womens.shape}")
print(f"  Season: {grid_stage2_womens.Season.unique()}")

Stage 1 prediction grid (women's): (258131, 4)
  Seasons: [2022 2023 2024 2025]
  Unique teams per season: {2022: 355, 2023: 360, 2024: 359, 2025: 361}

Stage 2 prediction grid (women's): (65703, 4)
  Season: [2026]


## 5. Merge Labels into Stage 1 Grid

For Stage 1 validation, we can check which matchups actually happened in the tournament and attach labels. Most matchups in the grid never happened (only ~63 games per tournament), but this lets us validate predictions against actual outcomes.

In [10]:
# Merge actual tournament results into Stage 1 grid
stage1_with_labels = grid_stage1_womens.merge(
    matchups[['ID', 'Label']],
    on='ID',
    how='left'
)

actual_games = stage1_with_labels['Label'].notna().sum()
print(f"Stage 1 grid rows: {len(stage1_with_labels):,}")
print(f"Rows with actual tournament results: {actual_games}")
print(f"Rows without results (never played): {len(stage1_with_labels) - actual_games:,}")

# Store this info on the grid
grid_stage1_womens = stage1_with_labels

Stage 1 grid rows: 258,131
Rows with actual tournament results: 268
Rows without results (never played): 257,863


## 6. Save Outputs

In [11]:
outputs = {
    'tourney_matchups': matchups,
    'prediction_grid_stage1': grid_stage1_womens,
    'prediction_grid_stage2': grid_stage2_womens,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/womens/tourney_matchups.parquet ((1717, 7))
Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/womens/prediction_grid_stage1.parquet ((258131, 5))
Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/womens/prediction_grid_stage2.parquet ((65703, 4))


## 7. Output Summary

In [12]:
print("=" * 60)
print("DATA SPLIT SUMMARY — WOMEN'S")
print("=" * 60)

print(f"\ntourney_matchups:")
print(f"  Shape: {matchups.shape}")
print(f"  Seasons: {matchups.Season.min()} - {matchups.Season.max()} (excl. 2020)")
print(f"  Total games: {len(matchups)}")
print(f"  Label balance: {matchups.Label.mean():.3f}")
print(f"  LOGO folds: {matchups.Fold.nunique()} seasons")
print(f"  Stage 1 validation games (2022-2025): {matchups.IsStage1Val.sum()}")

print(f"\nprediction_grid_stage1:")
print(f"  Shape: {grid_stage1_womens.shape}")
print(f"  Rows with labels: {grid_stage1_womens.Label.notna().sum()}")

print(f"\nprediction_grid_stage2:")
print(f"  Shape: {grid_stage2_womens.shape}")

print(f"\nCV STRATEGY: Leave-One-Season-Out (LOGO)")
print(f"  Each season is one fold.")
print(f"  Train on all other seasons, validate on the held-out season.")
print(f"  Final model comparison uses Stage 1 seasons (2022-2025).")
print(f"  2020 excluded (no tournament).")
print(f"  Women's tournament data starts 1998.")

DATA SPLIT SUMMARY — WOMEN'S

tourney_matchups:
  Shape: (1717, 7)
  Seasons: 1998 - 2025 (excl. 2020)
  Total games: 1717
  Label balance: 0.517
  LOGO folds: 27 seasons
  Stage 1 validation games (2022-2025): 268

prediction_grid_stage1:
  Shape: (258131, 5)
  Rows with labels: 268

prediction_grid_stage2:
  Shape: (65703, 4)

CV STRATEGY: Leave-One-Season-Out (LOGO)
  Each season is one fold.
  Train on all other seasons, validate on the held-out season.
  Final model comparison uses Stage 1 seasons (2022-2025).
  2020 excluded (no tournament).
  Women's tournament data starts 1998.
